# 03 — Link Prediction (Table 9 / Table 5)

Reproduces the AUC-ROC columns of **Table 9** (Appendix D.1, the fuller 7-dataset link-prediction table; Table 5 in the main text is a 4-dataset subset of the same numbers) via `main-link.py` (the Edge2Graph entrypoint). Each run trains its own encoder + MoE stage from scratch per `trainers/edge2graph_trainer.py::run()` and prints a final line:

```
[Test] Acc <acc> | AUC <auc> | Hits@<k> <hits>
```

**Prereq.** Run `00_setup.ipynb` in this same Colab session first.

In [ ]:
import os, sys, glob

# Self-sufficient by design: Colab typically gives each notebook its own fresh
# runtime, so REPO_DIR/DATA_DIR/etc from 00_setup.ipynb do NOT carry over unless
# you kept the exact same runtime connected. Re-declare the same paths here and
# fail loudly (with an actionable message) if the actual repo/build aren't
# present in *this* runtime, rather than silently trying to use a missing var.
REPO_DIR = '/content/R-GFM'
BASE = '/content/drive/MyDrive/R-GFM'
DATA_DIR = f'{BASE}/datasets'
CKPT_DIR = f'{BASE}/checkpoints'
RESULTS_DIR = f'{BASE}/results'

if not os.path.isdir(REPO_DIR) or not glob.glob(f'{REPO_DIR}/graph_aug/graph_aug_cuda*.so'):
    raise RuntimeError(
        "R-GFM repo / built CUDA extension not found in this runtime.\n"
        "Run 00_setup.ipynb FIRST in this exact runtime (Runtime -> Manage sessions "
        "to check if it's still connected). If this is a fresh runtime, 00_setup's "
        "clone + pip installs + CUDA build must be redone here — Drive contents "
        "persist across runtimes, but the cloned repo and build artifacts do not."
    )

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

from google.colab import drive
drive.mount('/content/drive')  # no-op if already mounted
for d in [DATA_DIR, CKPT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

LP_LOG_DIR = f'{RESULTS_DIR}/lp'
os.makedirs(LP_LOG_DIR, exist_ok=True)
print('Logs ->', LP_LOG_DIR)

In [ ]:
# wisconsin
LOG = f'{LP_LOG_DIR}/wisconsin.log'
!python main-link.py --dataset wisconsin --epochs 200 --device 0 \
    --dataset_dir "$DATA_DIR" --checkpoint_dir "$CKPT_DIR" --checkpoint_prefix edge2graph_wisconsin \
    2>&1 | tee $LOG

In [ ]:
# cornell
LOG = f'{LP_LOG_DIR}/cornell.log'
!python main-link.py --dataset cornell --epochs 200 --device 0 \
    --dataset_dir "$DATA_DIR" --checkpoint_dir "$CKPT_DIR" --checkpoint_prefix edge2graph_cornell \
    2>&1 | tee $LOG

In [ ]:
# citeseer
LOG = f'{LP_LOG_DIR}/citeseer.log'
!python main-link.py --dataset citeseer --epochs 200 --device 0 \
    --dataset_dir "$DATA_DIR" --checkpoint_dir "$CKPT_DIR" --checkpoint_prefix edge2graph_citeseer \
    2>&1 | tee $LOG

In [ ]:
# pubmed
LOG = f'{LP_LOG_DIR}/pubmed.log'
!python main-link.py --dataset pubmed --epochs 200 --device 0 \
    --dataset_dir "$DATA_DIR" --checkpoint_dir "$CKPT_DIR" --checkpoint_prefix edge2graph_pubmed \
    2>&1 | tee $LOG

In [ ]:
# cora
LOG = f'{LP_LOG_DIR}/cora.log'
!python main-link.py --dataset cora --epochs 200 --device 0 \
    --dataset_dir "$DATA_DIR" --checkpoint_dir "$CKPT_DIR" --checkpoint_prefix edge2graph_cora \
    2>&1 | tee $LOG

In [ ]:
# photo
LOG = f'{LP_LOG_DIR}/photo.log'
!python main-link.py --dataset photo --epochs 200 --device 0 \
    --dataset_dir "$DATA_DIR" --checkpoint_dir "$CKPT_DIR" --checkpoint_prefix edge2graph_photo \
    2>&1 | tee $LOG

In [ ]:
# texas
LOG = f'{LP_LOG_DIR}/texas.log'
!python main-link.py --dataset texas --epochs 200 --device 0 \
    --dataset_dir "$DATA_DIR" --checkpoint_dir "$CKPT_DIR" --checkpoint_prefix edge2graph_texas \
    2>&1 | tee $LOG

In [ ]:
# Quick summary — pull the final "[Test] Acc ... | AUC ... | Hits@..." line from each log.
import re
for ds in ['wisconsin', 'cornell', 'citeseer', 'pubmed', 'cora', 'photo', 'texas']:
    log = f'{LP_LOG_DIR}/{ds}.log'
    if not os.path.exists(log):
        print(f'{ds:12s}  (not run yet)')
        continue
    txt = open(log).read()
    m = re.findall(r'\[Test\]\s*Acc\s*([0-9.]+)\s*\|\s*AUC\s*([0-9.]+)\s*\|\s*Hits@(\d+)\s*([0-9.]+)', txt)
    if m:
        acc, auc, k, hits = m[-1]
        print(f'{ds:12s}  Acc {float(acc)*100:.2f}  AUC {float(auc)*100:.2f}  Hits@{k} {float(hits)*100:.2f}')
    else:
        print(f'{ds:12s}  (no result line found — check log)')

**Target numbers (paper's Table 9, R-GFM row, AUC-ROC %).**

| Wisconsin | Cornell | Citeseer | Pubmed | Cora | Photos | Texas |
|---:|---:|---:|---:|---:|---:|---:|
| 84.15 ± 0.71 | 85.90 ± 0.69 | 90.88 ± 0.70 | 88.62 ± 0.41 | 89.27 ± 0.64 | 81.53 ± 0.83 | 87.94 ± 0.96 |

Move on to `04_results_summary.ipynb` to build a combined paper-vs-ours comparison.